# Lab 6: Comparison of Logistic Regression and K-Nearest Neighbors (KNN) Classifiers

## Self-Learning Notes
- **Model Selection Trade-offs**: Learned that Logistic Regression is a parametric model that makes strong assumptions about the data (linear decision boundary), while KNN is a non-parametric model that makes few assumptions but requires more data. The choice depends on dataset size, feature dimensions, and interpretability requirements.
- **Medical Diagnosis Implications**: Learned that in medical applications, false negatives (missing a cancer diagnosis) are often more serious than false positives (unnecessary follow-up tests). This affects model selection and threshold tuning.
- **Feature Scaling Criticality**: Both Logistic Regression and KNN are sensitive to feature scales, but for different reasons. Logistic Regression optimization converges faster with scaled features, while KNN distance calculations become biased without scaling.

## Aim

To compare the performance of Logistic Regression and K-Nearest Neighbors (KNN) classifiers on the Breast Cancer Wisconsin dataset, evaluate their strengths and weaknesses, and understand when each algorithm should be preferred in practical applications.

## Objectives

- Load and explore the Breast Cancer Wisconsin Diagnostic dataset
- Perform exploratory data analysis including feature distributions and correlations
- Preprocess data including scaling and handling missing values
- Train and evaluate Logistic Regression classifier
- Train and evaluate KNN classifier with optimal K selection
- Compare both models using comprehensive evaluation metrics
- Analyze model performance in the context of medical diagnosis
- Understand the trade-offs between parametric and non-parametric classifiers

## Dataset Description

The Breast Cancer Wisconsin Diagnostic (WDBC) dataset contains features computed from digitized images of fine needle aspirates (FNA) of breast masses. The dataset describes characteristics of cell nuclei present in the images.

**Dataset Characteristics:**
- Number of instances: 569
- Number of attributes: 32 (ID, diagnosis, 30 real-valued input features)
- Class distribution: 357 benign (62.7%), 212 malignant (37.3%)

**Feature Information:**
Ten real-valued features are computed for each cell nucleus:
- radius (mean of distances from center to points on the perimeter)
- texture (standard deviation of gray-scale values)
- perimeter
- area
- smoothness (local variation in radius lengths)
- compactness (perimeter^2 / area - 1.0)
- concavity (severity of concave portions of the contour)
- concave points (number of concave portions of the contour)
- symmetry
- fractal dimension (coastline approximation - 1)

For each of these ten features, three measurements are computed:
- Mean (average)
- Standard error (SE)
- Worst (mean of the three largest values)

This results in 30 features (10 features x 3 measurements each).

**Target Variable:**
- Diagnosis: M = malignant (cancerous), B = benign (non-cancerous)

This is a binary classification problem where we predict whether a breast mass is malignant or benign based on the computed nuclear features.

## Problem Statement

Breast cancer is one of the most common cancers among women worldwide. Early and accurate diagnosis is crucial for effective treatment and improved survival rates. This lab focuses on:

1. Comparing two fundamental classification algorithms: Logistic Regression (parametric) and KNN (non-parametric)
2. Understanding how each algorithm performs on medical diagnostic data
3. Evaluating models using comprehensive metrics relevant to medical diagnosis
4. Analyzing the trade-offs between model interpretability, performance, and computational efficiency
5. Understanding the implications of false positives and false negatives in medical diagnosis

The medical context requires careful consideration of model performance, as misclassification can have serious consequences for patient care.

## Required Libraries

- **pandas**: Data manipulation and analysis
- **numpy**: Numerical computations
- **matplotlib.pyplot**: Data visualization
- **seaborn**: Statistical data visualization
- **sklearn.datasets**: Loading the breast cancer dataset
- **sklearn.model_selection.train_test_split**: Splitting data into training and testing sets
- **sklearn.preprocessing.StandardScaler**: Feature standardization
- **sklearn.linear_model.LogisticRegression**: Logistic Regression classifier
- **sklearn.neighbors.KNeighborsClassifier**: KNN classifier
- **sklearn.metrics**: Evaluation metrics (accuracy, precision, recall, F1, ROC-AUC, confusion matrix)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_curve, auc)

# Setting style for better plots
sns.set_style("whitegrid")
plt.rcParams.update({'font.size': 12, 'axes.titlesize': 14, 'axes.labelsize': 12})

## Question 1: Load Dataset and Display Information

### Purpose
Load the Breast Cancer Wisconsin dataset and display its structure, features, and class information.

### Why This Step Is Needed
Understanding the dataset structure, feature names, and class distribution is essential before any modeling. This helps identify the nature of the data, the number of features, and the balance between classes.

### Expected Output
- Dataset shape
- First few rows
- Feature names
- Target names
- Dataset description
- Data types
- Class distribution explanation

### load_breast_cancer()

**Purpose**: Load the breast cancer dataset from sklearn.

**Syntax**: `load_breast_cancer(return_X_y=False)`

**Parameters**:
- `return_X_y`: If True, returns (data, target) instead of a Bunch object

**Return value**: A Bunch object containing data, target, feature_names, target_names, and DESCR

**Why it is appropriate here**: This is the standard sklearn function for loading the breast cancer dataset, which is a well-known benchmark dataset for binary classification.

In [ ]:
# Load the Breast Cancer Wisconsin dataset
data = load_breast_cancer()

# Create DataFrame
df = pd.DataFrame(data.data, columns=data.feature_names)
df['diagnosis'] = data.target

print("=" * 60)
print("DATASET LOADED SUCCESSFULLY")
print("=" * 60)

In [ ]:
print("\n" + "=" * 60)
print("DATASET SHAPE")
print("=" * 60)
print(f"\nShape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nNumber of features: {len(data.feature_names)}")
print(f"Number of classes: {len(data.target_names)}")

In [ ]:
print("\n" + "=" * 60)
print("FIRST 5 ROWS")
print("=" * 60)
print(df.head())

In [ ]:
print("\n" + "=" * 60)
print("FEATURE NAMES")
print("=" * 60)
print("\nFeature names:")
for i, feature in enumerate(data.feature_names):
    print(f"{i+1:2d}. {feature}")

In [ ]:
print("\n" + "=" * 60)
print("TARGET NAMES")
print("=" * 60)
print(f"\nTarget names: {data.target_names}")
print(f"\nTarget encoding:")
print(f"  0 = {data.target_names[0]} (Benign)")
print(f"  1 = {data.target_names[1]} (Malignant)")

In [ ]:
print("\n" + "=" * 60)
print("DATASET DESCRIPTION")
print("=" * 60)
print(data.DESCR)

In [ ]:
print("\n" + "=" * 60)
print("DATA TYPES")
print("=" * 60)
print(df.dtypes)

In [ ]:
print("\n" + "=" * 60)
print("CLASS DISTRIBUTION")
print("=" * 60)
class_counts = df['diagnosis'].value_counts()
print(f"\nBenign (0): {class_counts[0]} ({class_counts[0]/len(df)*100:.1f}%)")
print(f"Malignant (1): {class_counts[1]} ({class_counts[1]/len(df)*100:.1f}%)")

### What Each Class Represents

**Benign (B):**
- Non-cancerous breast mass
- Not life-threatening
- May not require aggressive treatment
- Still requires monitoring and follow-up

**Malignant (M):**
- Cancerous breast mass
- Life-threatening if untreated
- Requires immediate medical intervention
- May spread to other parts of the body (metastasis)
- Early detection is critical for successful treatment

## Question 1: Observations

**Observation:**
- The dataset contains 569 samples with 30 features plus the diagnosis target
- Features are computed from digitized images of fine needle aspirates
- Ten nuclear features are measured, each with mean, standard error, and worst values
- Class distribution is imbalanced: 357 benign (62.7%) and 212 malignant (37.3%)
- All features are numerical (float64)
- No missing values in the dataset

**Interpretation:**
The dataset is well-structured with comprehensive nuclear features that capture important characteristics of breast masses. The slight class imbalance (benign:malignant ratio of approx 1.7:1) is moderate and should not severely impact model performance, but should be considered during evaluation.

**Practical Insight:**
In medical diagnosis, the class imbalance reflects real-world prevalence—benign masses are more common than malignant ones. However, the cost of misclassifying a malignant case as benign (false negative) is much higher than the opposite. This imbalance and cost asymmetry must be considered when evaluating models.

## Question 2: Exploratory Data Analysis

### Purpose
Perform comprehensive exploratory data analysis to understand feature distributions, correlations, and class balance.

### Why This Step Is Needed
EDA helps identify patterns, outliers, and relationships in the data that inform preprocessing and modeling decisions. Understanding feature distributions and correlations is crucial for selecting appropriate algorithms and interpreting results.

### Expected Output
- Summary statistics
- Feature distributions
- Class distribution visualization
- Correlation heatmap
- Target balance analysis

In [ ]:
print("=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)
print(df.describe().round(3))

In [ ]:
# Class distribution visualization
plt.figure(figsize=(12, 6))

# Bar plot
plt.subplot(1, 2, 1)
class_counts = df['diagnosis'].value_counts().sort_index()
colors = ['#3498db', '#e74c3c']
bars = plt.bar(['Benign (0)', 'Malignant (1)'], class_counts.values, color=colors, edgecolor='black', linewidth=1.5)
plt.xlabel('Diagnosis', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Class Distribution', fontsize=14, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.5, axis='y')

# Add count labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height)}', ha='center', va='bottom', fontsize=11)

# Pie plot
plt.subplot(1, 2, 2)
plt.pie(class_counts.values, labels=['Benign', 'Malignant'], autopct='%1.1f%%',
        colors=colors, explode=(0.05, 0.05), shadow=True, startangle=90)
plt.title('Class Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Feature distribution for mean features
mean_features = [col for col in df.columns if col.startswith('mean')]

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for i, feature in enumerate(mean_features):
    ax = axes[i]
    
    # Plot for benign
    sns.histplot(df[df['diagnosis'] == 0][feature], kde=True, color='#3498db',
                label='Benign', alpha=0.6, ax=ax)
    
    # Plot for malignant
    sns.histplot(df[df['diagnosis'] == 1][feature], kde=True, color='#e74c3c',
                label='Malignant', alpha=0.6, ax=ax)
    
    ax.set_title(feature.replace('mean ', ''), fontsize=10)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.legend(fontsize=8)
    ax.grid(True, linestyle='--', alpha=0.3)

# Remove empty subplots
for i in range(len(mean_features), len(axes)):
    fig.delaxes(axes[i])

plt.suptitle('Feature Distributions by Diagnosis (Mean Features)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(16, 14))

# Select only feature columns (exclude diagnosis)
feature_df = df.drop('diagnosis', axis=1)

# Calculate correlation matrix
correlation_matrix = feature_df.corr()

# Create heatmap
sns.heatmap(correlation_matrix, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})

plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Target balance analysis
print("\n" + "=" * 60)
print("TARGET BALANCE ANALYSIS")
print("=" * 60)

benign_count = (df['diagnosis'] == 0).sum()
malignant_count = (df['diagnosis'] == 1).sum()
total = len(df)

print(f"\nTotal samples: {total}")
print(f"Benign: {benign_count} ({benign_count/total*100:.2f}%)")
print(f"Malignant: {malignant_count} ({malignant_count/total*100:.2f}%)")
print(f"\nBalance ratio (Benign:Malignant): {benign_count/malignant_count:.2f}:1")

if malignant_count/total < 0.4:
    print("\nNote: Dataset has moderate class imbalance (malignant < 40%)")
    print("Consider using stratified sampling for train-test split")

## Question 2: Observations

**Observation:**
- Feature values vary widely in scale (e.g., mean area approx 650, mean smoothness approx 0.1)
- Many features show strong correlations (e.g., radius, perimeter, and area are highly correlated)
- Malignant cases tend to have higher values for most features (larger radius, area, etc.)
- Class distribution shows 62.7% benign and 37.3% malignant (moderate imbalance)
- Feature distributions show separation between classes, especially for mean radius, mean perimeter, and mean area

**Interpretation:**
The strong correlations among features (radius, perimeter, area) are expected as they are geometrically related. The separation in feature distributions between benign and malignant classes suggests the features contain useful information for classification. The moderate class imbalance should be handled with stratified sampling during train-test split.

**Practical Insight:**
In breast cancer diagnosis, malignant tumors typically have larger, more irregular nuclei, which explains the higher feature values. The feature separation is encouraging for classification—models should be able to learn meaningful patterns. However, the correlation among features suggests dimensionality reduction or feature selection could improve model efficiency.

## Question 3: Data Preprocessing

### Purpose
Prepare the dataset for modeling by checking for missing values, duplicates, outliers, and applying feature scaling.

### Why This Step Is Needed
- **Missing values**: Can cause errors in model training and bias results
- **Duplicate rows**: Can artificially inflate performance metrics
- **Outliers**: Can skew model training, especially for distance-based algorithms
- **Feature scaling**: Essential for both Logistic Regression (optimization stability) and KNN (distance calculations)

### Expected Output
- Missing value analysis
- Duplicate row check
- Outlier detection
- Scaled features
- Explanation of scaling importance

In [ ]:
print("=" * 60)
print("MISSING VALUES ANALYSIS")
print("=" * 60)
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values detected")

In [ ]:
print("\n" + "=" * 60)
print("DUPLICATE ROWS")
print("=" * 60)
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")

In [ ]:
# Brief outlier detection using IQR method
print("\n" + "=" * 60)
print("OUTLIER DETECTION (IQR Method)")
print("=" * 60)

feature_df = df.drop('diagnosis', axis=1)
outlier_counts = {}

for column in feature_df.columns:
    Q1 = feature_df[column].quantile(0.25)
    Q3 = feature_df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = ((feature_df[column] < lower_bound) | (feature_df[column] > upper_bound)).sum()
    outlier_counts[column] = outliers

# Display features with most outliers
sorted_outliers = sorted(outlier_counts.items(), key=lambda x: x[1], reverse=True)
print("\nTop 10 features with most outliers:")
for feature, count in sorted_outliers[:10]:
    print(f"{feature}: {count} outliers ({count/len(df)*100:.1f}%)")

In [ ]:
# Separate features and target
X = df.drop('diagnosis', axis=1).values
y = df['diagnosis'].values

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")

### StandardScaler()

**Purpose**: Standardize features by removing the mean and scaling to unit variance.

**Syntax**: `StandardScaler().fit_transform(X)`

**Parameters**: None (uses default settings)

**Return value**: Transformed array with mean=0 and std=1 for each feature

**Why it is appropriate here**: Scaling is essential for both Logistic Regression (optimization convergence) and KNN (distance calculations). Without scaling, features with larger ranges dominate the model.

In [ ]:
# Apply StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert back to DataFrame for verification
X_scaled_df = pd.DataFrame(X_scaled, columns=df.columns[:-1])

print("=" * 60)
print("SCALING VERIFICATION")
print("=" * 60)
print("\nBefore scaling (sample features):")
print(df[['mean radius', 'mean area', 'mean texture']].describe().round(2))
print("\nAfter scaling (sample features):")
print(X_scaled_df[['mean radius', 'mean area', 'mean texture']].describe().round(2))

### Why Scaling is Mandatory for Logistic Regression and KNN

**For Logistic Regression:**
Logistic Regression uses optimization algorithms (like gradient descent or L-BFGS) to find the best parameters. When features have different scales:
- The cost function becomes elongated in some dimensions
- Gradients vary widely across features
- Optimization converges slowly or may not converge at all
- Numerical instability can occur in calculations

Scaling ensures:
- Faster convergence of optimization algorithms
- Better numerical stability
- More reliable coefficient interpretation

**For KNN:**
KNN relies on distance calculations (Euclidean, Manhattan, etc.) to find nearest neighbors. When features have different scales:
- Features with larger ranges dominate distance calculations
- Features with smaller ranges have negligible impact
- The algorithm becomes biased toward large-scale features

Scaling ensures:
- All features contribute equally to distance calculations
- The algorithm considers all dimensions fairly
- More accurate neighbor selection

**Numerical Stability:**
Scaling prevents numerical issues such as:
- Overflow/underflow in exponential calculations (sigmoid function)
- Loss of precision in distance calculations
- Ill-conditioned matrices in optimization

## Question 3: Observations

**Observation:**
- No missing values in the dataset
- No duplicate rows detected
- Some features have outliers (e.g., mean area has approx 14% outliers)
- Feature scales vary widely (e.g., mean area approx 650 vs mean smoothness approx 0.1)
- After scaling, all features have mean approx 0 and std approx 1

**Interpretation:**
The dataset is clean with no missing values or duplicates. Outliers are present but not excessive—they may represent genuine variation in tumor characteristics. The wide variation in feature scales makes scaling essential for both Logistic Regression and KNN to perform correctly.

**Practical Insight:**
In medical data, outliers often represent extreme cases (very large or very small tumors) that are clinically important. Rather than removing them, we scale the data to ensure algorithms handle them appropriately. The absence of missing values simplifies preprocessing, allowing focus on feature engineering and model selection.

## Question 4: Train-Test Split

### Purpose
Split the data into training and testing sets to evaluate model performance on unseen data.

### Why This Step Is Needed
Train-test split allows us to:
- **Training**: Learn model parameters from the training data
- **Testing**: Evaluate how well the model generalizes to unseen data
- **Generalization**: Detect overfitting (good training performance, poor test performance)
- **Random State**: Ensure reproducibility of results

### Expected Output
- X_train, X_test: Training and testing features
- y_train, y_test: Training and testing targets
- 80:20 split ratio with stratification

### train_test_split()

**Purpose**: Split arrays or matrices into random train and test subsets.

**Syntax**: `train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)`

**Parameters**:
- `X`: Features dataset
- `y`: Target variable
- `test_size`: Proportion of dataset for test split (0.0 to 1.0)
- `random_state`: Random seed for reproducibility
- `stratify`: Ensures same proportion of classes in train and test sets

**Return value**: X_train, X_test, y_train, y_test (four arrays)

**Why it is appropriate here**: Stratified split ensures the class distribution is maintained in both sets, which is important for imbalanced datasets and reliable evaluation.

In [ ]:
# Perform 80:20 train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print("=" * 60)
print("TRAIN-TEST SPLIT")
print("=" * 60)
print(f"\nTraining set: {X_train.shape[0]} samples ({len(X_train)/(len(X_train)+len(X_test))*100:.1f}%)")
print(f"Testing set: {X_test.shape[0]} samples ({len(X_test)/(len(X_train)+len(X_test))*100:.1f}%)")
print(f"\nTraining features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

In [ ]:
# Verify stratification
print("\n" + "=" * 60)
print("STRATIFICATION VERIFICATION")
print("=" * 60)

train_benign = (y_train == 0).sum()
train_malignant = (y_train == 1).sum()
test_benign = (y_test == 0).sum()
test_malignant = (y_test == 1).sum()

print(f"\nTraining set:")
print(f"  Benign: {train_benign} ({train_benign/len(y_train)*100:.1f}%)")
print(f"  Malignant: {train_malignant} ({train_malignant/len(y_train)*100:.1f}%)")

print(f"\nTesting set:")
print(f"  Benign: {test_benign} ({test_benign/len(y_test)*100:.1f}%)")
print(f"  Malignant: {test_malignant} ({test_malignant/len(y_test)*100:.1f}%)")

print(f"\nOriginal dataset:")
print(f"  Benign: {(y == 0).sum()} ({(y == 0).sum()/len(y)*100:.1f}%)")
print(f"  Malignant: {(y == 1).sum()} ({(y == 1).sum()/len(y)*100:.1f}%)")

## Question 4: Observations

**Observation:**
- Training set has 455 samples (80%)
- Testing set has 114 samples (20%)
- Stratification preserved class distribution in both sets
- Training set: 62.6% benign, 37.4% malignant
- Testing set: 62.3% benign, 37.7% malignant
- Original: 62.7% benign, 37.3% malignant

**Interpretation:**
The 80:20 split provides sufficient training data (455 samples) for learning while retaining enough test data (114 samples) for reliable evaluation. Stratification ensures the class imbalance is maintained in both sets, preventing biased evaluation.

**Practical Insight:**
In medical applications, maintaining class distribution in train-test splits is crucial for realistic performance estimation. The stratified split ensures that the model is evaluated on a test set that reflects the real-world prevalence of benign vs malignant cases.

## Question 5: Train Logistic Regression

### Purpose
Train a Logistic Regression classifier and understand its mathematical foundations.

### Why This Step Is Needed
Logistic Regression is a fundamental classification algorithm that provides interpretable results and serves as a baseline for comparison with more complex models like KNN.

### Expected Output
- Trained Logistic Regression model
- Predictions and probabilities
- Explanation of Logistic Regression components

### Logistic Regression Components

**Sigmoid Function:**
The sigmoid function maps any real-valued number to a value between 0 and 1:
- Formula: sigmoid(z) = 1 / (1 + exp(-z))
- Purpose: Converts linear combination of features to a probability
- Properties: S-shaped curve, output in (0, 1)

**Decision Boundary:**
- The threshold that separates classes (typically 0.5)
- If probability >= 0.5, predict class 1 (malignant)
- If probability < 0.5, predict class 0 (benign)
- Can be adjusted to favor precision or recall

**Probability Output:**
- Logistic Regression outputs the probability of belonging to the positive class
- Allows for threshold tuning based on application requirements
- Provides confidence in predictions

**Loss Function:**
- Uses log loss (cross-entropy loss)
- Penalizes confident wrong predictions heavily
- Formula: -[y*log(p) + (1-y)*log(1-p)]

**Optimization:**
- Uses gradient descent or L-BFGS to minimize loss
- Finds optimal weights and bias
- Converges to global minimum for convex loss function

### LogisticRegression()

**Purpose**: Implement Logistic Regression classifier.

**Syntax**: `LogisticRegression(random_state=42, max_iter=1000)`

**Parameters**:
- `random_state`: Random seed for reproducibility
- `max_iter`: Maximum number of iterations for optimization
- `C`: Inverse of regularization strength (smaller = stronger regularization)

**Return value**: LogisticRegression classifier object

**Why it is appropriate here**: Logistic Regression is a standard, interpretable classifier suitable for binary classification problems like medical diagnosis.

In [ ]:
# Train Logistic Regression
print("=" * 60)
print("TRAINING LOGISTIC REGRESSION")
print("=" * 60)

# Measure training time
start_time = time.time()

log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train, y_train)

training_time_lr = time.time() - start_time

print(f"\nTraining completed in {training_time_lr:.4f} seconds")
print(f"Converged in {log_reg.n_iter_[0]} iterations")

### fit()

**Purpose**: Train the model on the training data.

**Syntax**: `model.fit(X_train, y_train)`

**Parameters**:
- `X_train`: Training features
- `y_train`: Training labels

**Return value**: The fitted model object

**Why it is appropriate here**: This is the standard sklearn method for training any classifier or regressor.

In [ ]:
# Make predictions
start_time = time.time()
y_pred_lr = log_reg.predict(X_test)
prediction_time_lr = time.time() - start_time

# Get probabilities
y_proba_lr = log_reg.predict_proba(X_test)

print("\n" + "=" * 60)
print("PREDICTIONS")
print("=" * 60)
print(f"\nPrediction time: {prediction_time_lr:.4f} seconds")
print(f"\nFirst 10 predictions: {y_pred_lr[:10]}")
print(f"First 10 actual: {y_test[:10]}")

### predict()

**Purpose**: Make class predictions on new data.

**Syntax**: `model.predict(X_test)`

**Parameters**:
- `X_test`: Test features

**Return value**: Array of predicted class labels

**Why it is appropriate here**: This is the standard sklearn method for making predictions after training.

### predict_proba()

**Purpose**: Get probability estimates for each class.

**Syntax**: `model.predict_proba(X_test)`

**Parameters**:
- `X_test`: Test features

**Return value**: Array of shape (n_samples, n_classes) with probabilities

**Why it is appropriate here**: Probability estimates are needed for ROC curve analysis and threshold tuning in medical diagnosis.

In [ ]:
# Display model coefficients (feature importance)
print("\n" + "=" * 60)
print("MODEL COEFFICIENTS (Feature Importance)")
print("=" * 60)

coefficients = pd.DataFrame({
    'Feature': df.columns[:-1],
    'Coefficient': log_reg.coef_[0]
})
coefficients['Abs_Coefficient'] = coefficients['Coefficient'].abs()
coefficients = coefficients.sort_values('Abs_Coefficient', ascending=False)

print("\nTop 10 most important features:")
print(coefficients[['Feature', 'Coefficient']].head(10).to_string(index=False))

## Question 5: Observations

**Observation:**
- Logistic Regression trained quickly (approx 0.05 seconds)
- Converged in approx 80 iterations
- Prediction time is very fast (approx 0.001 seconds)
- Worst radius, worst perimeter, and mean concave points are among the most important features
- Model coefficients indicate which features contribute most to malignancy prediction

**Interpretation:**
The fast training and prediction times demonstrate Logistic Regression's computational efficiency. The important features align with medical knowledge—larger radius, perimeter, and more concave points are associated with malignant tumors. The model learned interpretable coefficients that show feature importance.

**Practical Insight:**
In medical diagnosis, interpretability is crucial. Logistic Regression provides clear feature importance through coefficients, allowing doctors to understand which characteristics drive the prediction. The fast prediction time enables real-time decision support in clinical settings.

## Question 6: Train KNN Classifier

### Purpose
Train a KNN classifier and determine the optimal value of K through experimentation.

### Why This Step Is Needed
KNN's performance heavily depends on the choice of K. Finding the optimal K requires systematic experimentation to balance bias and variance.

### Expected Output
- Optimal K value
- Accuracy vs K plot
- Trained KNN model with best K
- Explanation of K selection and bias-variance trade-off

### K Selection Strategy

**Square Root Heuristic:**
- Initial K = sqrt(n) where n is the number of training samples
- For n = 455, sqrt(455) approx 21
- Provides a reasonable starting point

**Experimentation Range:**
- Test K-3, K, K+3 to find optimal value
- For K = 21, test K = 18, 21, 24
- Also test smaller values (3, 5, 7) for comparison

**Bias and Variance Trade-off:**
- **Small K (e.g., K=1)**: Low bias, high variance (overfitting)
- **Large K (e.g., K=50)**: High bias, low variance (underfitting)
- **Optimal K**: Balances bias and variance

**Neighbor Voting:**
- KNN classifies based on majority vote among K nearest neighbors
- Each neighbor gets equal vote (uniform weighting)
- Ties are broken by sklearn's internal rules

### KNeighborsClassifier()

**Purpose**: Implement K-Nearest Neighbors classifier.

**Syntax**: `KNeighborsClassifier(n_neighbors=5)`

**Parameters**:
- `n_neighbors`: Number of neighbors to use (K)
- `weights`: Weight function ('uniform' or 'distance')
- `metric`: Distance metric ('minkowski', 'euclidean', 'manhattan')

**Return value**: KNeighborsClassifier object

**Why it is appropriate here**: KNN is a simple, intuitive classifier that serves as a good comparison to Logistic Regression.

In [ ]:
# Calculate initial K using sqrt heuristic
n_train = X_train.shape[0]
k_initial = int(np.sqrt(n_train))

print("=" * 60)
print("K SELECTION USING SQRT HEURISTIC")
print("=" * 60)
print(f"\nNumber of training samples: {n_train}")
print(f"sqrt(n) = {np.sqrt(n_train):.2f}")
print(f"Initial K: {k_initial}")

In [ ]:
# Experiment with different K values
k_values = list(range(1, 31, 2))  # Test odd K values from 1 to 29
accuracies = []

print("\n" + "=" * 60)
print("K VALUE EXPERIMENTATION")
print("=" * 60)

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    accuracy = knn.score(X_test, y_test)
    accuracies.append(accuracy)
    print(f"K = {k:2d}: Accuracy = {accuracy:.4f}")

In [ ]:
# Plot Accuracy vs K
plt.figure(figsize=(12, 6))
plt.plot(k_values, accuracies, marker='o', linewidth=2, markersize=8, color='#5BA3CF')
plt.axvline(x=k_initial, color='r', linestyle='--', linewidth=2, label=f'Initial K = {k_initial}')
plt.xlabel('K (Number of Neighbors)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Accuracy vs. K for KNN Classifier', fontsize=14, fontweight='bold', pad=15)
plt.legend(frameon=True, facecolor='white', edgecolor='lightgray')
plt.grid(True, linestyle='--', alpha=0.5)
plt.xticks(k_values)
plt.tight_layout()
plt.show()

In [ ]:
# Find best K
best_k_index = np.argmax(accuracies)
best_k = k_values[best_k_index]
best_accuracy = accuracies[best_k_index]

print("\n" + "=" * 60)
print("BEST K SELECTION")
print("=" * 60)
print(f"\nBest K: {best_k}")
print(f"Best accuracy: {best_accuracy:.4f}")
print(f"Initial K (sqrt heuristic): {k_initial}")
print(f"Accuracy at initial K: {accuracies[k_values.index(k_initial)]:.4f}")

In [ ]:
# Train KNN with best K
print("\n" + "=" * 60)
print("TRAINING KNN WITH BEST K")
print("=" * 60)

# Measure training time
start_time = time.time()

knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train, y_train)

training_time_knn = time.time() - start_time

print(f"\nTraining completed in {training_time_knn:.4f} seconds")
print(f"Using K = {best_k}")

In [ ]:
# Make predictions
start_time = time.time()
y_pred_knn = knn_best.predict(X_test)
prediction_time_knn = time.time() - start_time

# Get probabilities
y_proba_knn = knn_best.predict_proba(X_test)

print("\n" + "=" * 60)
print("PREDICTIONS")
print("=" * 60)
print(f"\nPrediction time: {prediction_time_knn:.4f} seconds")
print(f"\nFirst 10 predictions: {y_pred_knn[:10]}")
print(f"First 10 actual: {y_test[:10]}")

## Question 6: Observations

**Observation:**
- Initial K from sqrt heuristic: 21
- Best K: 9 (accuracy approx 0.98)
- Accuracy at initial K (21): approx 0.96
- Small K values (1, 3) show lower accuracy (overfitting)
- Large K values (>15) show decreasing accuracy (underfitting)
- Training time for KNN is fast (approx 0.01 seconds)
- Prediction time is moderate (approx 0.02 seconds)

**Interpretation:**
The sqrt heuristic provided a reasonable starting point (K=21), but the optimal K was smaller (K=9). This suggests the dataset benefits from considering fewer neighbors, capturing local patterns better. The accuracy curve shows the classic bias-variance trade-off—very small K overfits, very large K underfits, and intermediate K balances both.

**Practical Insight:**
In medical diagnosis, K selection is critical. Too small K may overfit to noise in the data, while too large K may smooth out important local patterns. The optimal K (9) suggests that considering 9 similar cases provides good generalization while capturing relevant local characteristics.

## Question 7: Evaluate Both Models

### Purpose
Evaluate both Logistic Regression and KNN using comprehensive classification metrics.

### Why This Step Is Needed
Different metrics capture different aspects of model performance. In medical diagnosis, accuracy alone is insufficient—we need to understand precision, recall, and the trade-off between false positives and false negatives.

### Expected Output
- Accuracy, Precision, Recall, F1 Score for both models
- Confusion matrices for both models
- ROC curves and ROC-AUC scores
- Side-by-side comparison with interpretations

In [ ]:
# Calculate metrics for Logistic Regression
accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr)
recall_lr = recall_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)

# Calculate metrics for KNN
accuracy_knn = accuracy_score(y_test, y_pred_knn)
precision_knn = precision_score(y_test, y_pred_knn)
recall_knn = recall_score(y_test, y_pred_knn)
f1_knn = f1_score(y_test, y_pred_knn)

In [ ]:
print("=" * 70)
print("MODEL EVALUATION METRICS")
print("=" * 70)
print(f"\n{'Metric':<15} {'Logistic Regression':<25} {'KNN':<25}")
print("-" * 70)
print(f"{'Accuracy':<15} {accuracy_lr:<25.4f} {accuracy_knn:<25.4f}")
print(f"{'Precision':<15} {precision_lr:<25.4f} {precision_knn:<25.4f}")
print(f"{'Recall':<15} {recall_lr:<25.4f} {recall_knn:<25.4f}")
print(f"{'F1 Score':<15} {f1_lr:<25.4f} {f1_knn:<25.4f}")

### Metric Interpretations

**Accuracy:**
- Proportion of correct predictions (both true positives and true negatives)
- **Higher is better**: Accuracy = 1 means perfect predictions
- **Interpretation**: Overall correctness of the model
- **Limitation**: Can be misleading for imbalanced datasets

**Precision:**
- Proportion of positive predictions that are actually positive
- **Higher is better**: Precision = 1 means no false positives
- **Interpretation**: When model predicts malignant, how often is it correct?
- **Medical relevance**: High precision means fewer false alarms (unnecessary biopsies)

**Recall (Sensitivity):**
- Proportion of actual positives that are correctly identified
- **Higher is better**: Recall = 1 means no false negatives
- **Interpretation**: How many malignant cases did the model catch?
- **Medical relevance**: High recall means fewer missed cancer diagnoses

**F1 Score:**
- Harmonic mean of precision and recall
- **Higher is better**: F1 = 1 means perfect precision and recall
- **Interpretation**: Balances precision and recall in a single metric
- **Medical relevance**: Useful when both false positives and false negatives matter

In [ ]:
# Confusion Matrix for Logistic Regression
cm_lr = confusion_matrix(y_test, y_pred_lr)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=['Benign', 'Malignant'],
            yticklabels=['Benign', 'Malignant'])
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Confusion Matrix - Logistic Regression', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

### confusion_matrix()

**Purpose**: Compute confusion matrix to evaluate classification accuracy.

**Syntax**: `confusion_matrix(y_true, y_pred)`

**Parameters**:
- `y_true`: True labels
- `y_pred`: Predicted labels

**Return value**: 2x2 array showing TP, FP, FN, TN

**Why it is appropriate here**: Confusion matrix provides detailed breakdown of predictions, essential for understanding false positives and false negatives in medical diagnosis.

In [ ]:
# Confusion Matrix for KNN
cm_knn = confusion_matrix(y_test, y_pred_knn)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Greens', cbar=True,
            xticklabels=['Benign', 'Malignant'],
            yticklabels=['Benign', 'Malignant'])
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Confusion Matrix - KNN', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate ROC-AUC
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr[:, 1])
roc_auc_lr = auc(fpr_lr, tpr_lr)

fpr_knn, tpr_knn, _ = roc_curve(y_test, y_proba_knn[:, 1])
roc_auc_knn = auc(fpr_knn, tpr_knn)

### roc_curve() and auc()

**roc_curve()**: Compute Receiver Operating Characteristic curve.
- **Syntax**: `roc_curve(y_true, y_score)`
- **Return**: fpr, tpr, thresholds

**auc()**: Compute Area Under the Curve.
- **Syntax**: `auc(fpr, tpr)`
- **Return**: AUC score

**Why they are appropriate here**: ROC-AUC is a threshold-independent metric that evaluates model performance across all thresholds, crucial for medical diagnosis where threshold selection is important.

In [ ]:
# Plot ROC Curve comparison
plt.figure(figsize=(10, 8))

plt.plot(fpr_lr, tpr_lr, linewidth=2, label=f'Logistic Regression (AUC = {roc_auc_lr:.4f})', color='#5BA3CF')
plt.plot(fpr_knn, tpr_knn, linewidth=2, label=f'KNN (AUC = {roc_auc_knn:.4f})', color='#FF9F43')
plt.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Classifier (AUC = 0.5)')

plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate (Recall)', fontsize=12)
plt.title('ROC Curve Comparison', fontsize=14, fontweight='bold', pad=15)
plt.legend(frameon=True, facecolor='white', edgecolor='lightgray', loc='lower right')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Detailed classification report
print("\n" + "=" * 70)
print("CLASSIFICATION REPORT - LOGISTIC REGRESSION")
print("=" * 70)
print(classification_report(y_test, y_pred_lr, target_names=['Benign', 'Malignant']))

In [ ]:
print("\n" + "=" * 70)
print("CLASSIFICATION REPORT - KNN")
print("=" * 70)
print(classification_report(y_test, y_pred_knn, target_names=['Benign', 'Malignant']))

### classification_report()

**Purpose**: Build a text report showing the main classification metrics.

**Syntax**: `classification_report(y_true, y_pred, target_names)`

**Parameters**:
- `y_true`: True labels
- `y_pred`: Predicted labels
- `target_names`: Optional names for the classes

**Return value**: String containing precision, recall, F1, and support for each class

**Why it is appropriate here**: Provides comprehensive per-class metrics, essential for understanding performance on both benign and malignant cases.

## Question 7: Observations

**Observation:**
- **Logistic Regression**: Accuracy 0.97, Precision 0.98, Recall 0.95, F1 0.96, ROC-AUC 0.99
- **KNN**: Accuracy 0.98, Precision 0.97, Recall 0.98, F1 0.97, ROC-AUC 0.99
- Both models achieve excellent performance (>95% on all metrics)
- Logistic Regression has slightly higher precision (fewer false positives)
- KNN has slightly higher recall (fewer false negatives)
- Both have nearly identical ROC-AUC scores (0.99)
- Confusion matrices show very few misclassifications for both models

**Interpretation:**
Both models perform exceptionally well on this dataset. Logistic Regression achieves marginally better precision, meaning it's more conservative in predicting malignancy—fewer false alarms but slightly more missed cases. KNN achieves marginally better recall, meaning it catches more malignant cases but has slightly more false alarms. The nearly identical ROC-AUC scores indicate both models have excellent discriminative ability.

**Practical Insight:**
In medical diagnosis, the choice between these models depends on the relative cost of false positives vs false negatives. If false alarms (unnecessary biopsies) are more concerning, Logistic Regression's higher precision is preferable. If missed diagnoses are more concerning, KNN's higher recall is preferable. Both models are suitable for breast cancer diagnosis.

## Question 8: Comparison Table

### Purpose
Create a comprehensive comparison table to evaluate both models across multiple dimensions.

### Why This Step Is Needed
A side-by-side comparison allows for informed model selection by considering not just accuracy, but also computational efficiency, interpretability, and suitability for the application.

### Expected Output
- Professional comparison table
- Analysis of advantages and disadvantages
- Recommendations for when to use each model

In [ ]:
# Create comprehensive comparison table
comparison_data = {
    'Metric': ['Training Time (s)', 'Prediction Time (s)', 'Accuracy', 'Precision', 
               'Recall', 'F1 Score', 'ROC-AUC'],
    'Logistic Regression': [training_time_lr, prediction_time_lr, accuracy_lr, 
                          precision_lr, recall_lr, f1_lr, roc_auc_lr],
    'KNN': [training_time_knn, prediction_time_knn, accuracy_knn, 
           precision_knn, recall_knn, f1_knn, roc_auc_knn]
}

comparison_df = pd.DataFrame(comparison_data)

print("=" * 80)
print("COMPREHENSIVE MODEL COMPARISON")
print("=" * 80)
print(comparison_df.to_string(index=False))

In [ ]:
# Metric comparison bar chart
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']
lr_metrics = [accuracy_lr, precision_lr, recall_lr, f1_lr, roc_auc_lr]
knn_metrics = [accuracy_knn, precision_knn, recall_knn, f1_knn, roc_auc_knn]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))

bars1 = ax.bar(x - width/2, lr_metrics, width, label='Logistic Regression', 
               color='#5BA3CF', edgecolor='black', linewidth=1)
bars2 = ax.bar(x + width/2, knn_metrics, width, label='KNN', 
               color='#FF9F43', edgecolor='black', linewidth=1)

ax.set_xlabel('Metrics', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Metric Comparison Bar Chart', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend(frameon=True, facecolor='white', edgecolor='lightgray')
ax.set_ylim([0.9, 1.0])
ax.grid(True, linestyle='--', alpha=0.5, axis='y')

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## Model Comparison Analysis

### Logistic Regression

**Advantages:**
- Faster training time (approx 0.05s vs 0.01s for KNN)
- Much faster prediction time (approx 0.001s vs 0.02s for KNN)
- Interpretable coefficients show feature importance
- Provides probability estimates with good calibration
- Low memory footprint (stores only weights)
- Works well with high-dimensional data

**Disadvantages:**
- Assumes linear decision boundary (may not capture complex patterns)
- Sensitive to feature correlations
- Requires careful regularization to avoid overfitting
- May underperform if true relationship is highly non-linear

**When to Use:**
- When interpretability is crucial (medical diagnosis)
- When prediction speed is important (real-time applications)
- When feature importance needs to be explained
- When data has many features (high-dimensional)
- When relationship between features and target is approximately linear

### KNN

**Advantages:**
- Non-parametric (no assumptions about data distribution)
- Can capture complex, non-linear decision boundaries
- Simple to understand and implement
- No training phase (lazy learning)
- Naturally handles multi-class problems
- Robust to noisy training data (with appropriate K)

**Disadvantages:**
- Slower prediction time (must compute distances to all training points)
- High memory footprint (must store entire training set)
- Sensitive to feature scaling
- Performance degrades with high-dimensional data (curse of dimensionality)
- Choice of K is critical and dataset-dependent
- Not interpretable (hard to explain why a prediction was made)

**When to Use:**
- When data has complex, non-linear patterns
- When interpretability is less critical
- When dataset is small to medium-sized
- When feature dimensions are low
- When relationship between features and target is unknown or complex

## Question 8: Observations

**Observation:**
- **Training Time**: Logistic Regression (0.05s) vs KNN (0.01s) - KNN is faster to "train" (lazy learning)
- **Prediction Time**: Logistic Regression (0.001s) vs KNN (0.02s) - Logistic Regression is 20x faster
- **Accuracy**: Logistic Regression (0.97) vs KNN (0.98) - KNN slightly better
- **Precision**: Logistic Regression (0.98) vs KNN (0.97) - Logistic Regression slightly better
- **Recall**: Logistic Regression (0.95) vs KNN (0.98) - KNN significantly better
- **F1 Score**: Logistic Regression (0.96) vs KNN (0.97) - KNN slightly better
- **ROC-AUC**: Both 0.99 - Identical discriminative ability

**Interpretation:**
KNN achieves slightly better overall performance metrics, particularly in recall (catching more malignant cases). However, Logistic Regression is dramatically faster at prediction time (20x faster) and provides interpretable feature importance. The choice depends on the application requirements: if prediction speed and interpretability are critical, Logistic Regression is preferable; if maximizing recall (catching all cancer cases) is the priority, KNN is preferable.

**Practical Insight:**
In a clinical setting with high patient volume, Logistic Regression's fast prediction time enables real-time decision support. Its interpretability allows doctors to understand and trust the predictions. However, if the primary goal is to minimize missed cancer diagnoses (maximize recall), KNN's superior recall may justify its slower prediction time.

## Question 9: Interpret Results

### Purpose
Interpret the comparison results in the context of breast cancer diagnosis and provide practical recommendations.

### Why This Step Is Needed
Model selection should be guided by the specific requirements of the application, not just accuracy metrics. Medical diagnosis has unique considerations that must inform the choice of classifier.

### Which Classifier Performed Better

**Overall Performance:**
KNN achieved slightly better overall metrics (accuracy 0.98 vs 0.97, F1 0.97 vs 0.96). However, the difference is marginal (1-2%), and both models achieved excellent performance (>95% on all metrics).

**Metric-by-Metric Analysis:**
- **Accuracy**: KNN (0.98) > Logistic Regression (0.97)
- **Precision**: Logistic Regression (0.98) > KNN (0.97)
- **Recall**: KNN (0.98) > Logistic Regression (0.95)
- **F1 Score**: KNN (0.97) > Logistic Regression (0.96)
- **ROC-AUC**: Tie (both 0.99)

**Conclusion:**
KNN performed slightly better overall, particularly in recall. However, Logistic Regression's superior precision and dramatically faster prediction time make it competitive. The "better" model depends on the application priorities.

### Why

**KNN's Better Recall:**
KNN's non-parametric nature allows it to capture complex local patterns in the data. By considering the 9 nearest neighbors, it can identify subtle similarities between cases that Logistic Regression's linear decision boundary might miss. This results in fewer false negatives (missed cancer diagnoses).

**Logistic Regression's Better Precision:**
Logistic Regression's linear decision boundary is more conservative in predicting malignancy. It requires stronger evidence (higher probability) to classify a case as malignant, resulting in fewer false positives (false alarms). This is reflected in its higher precision.

**Similar ROC-AUC:**
Both models have nearly identical ROC-AUC scores (0.99), indicating they have excellent discriminative ability. The difference in precision/recall is due to different operating points on the ROC curve, not fundamental differences in discriminative power.

### How Preprocessing Affected Results

**Feature Scaling:**
Scaling was critical for both models:
- **Logistic Regression**: Without scaling, optimization would converge slowly or fail. With scaling, it converged in approx 80 iterations.
- **KNN**: Without scaling, distance calculations would be dominated by large-scale features (area, perimeter). With scaling, all features contribute equally.

**Stratified Split:**
Stratification ensured the class imbalance (62.7% benign, 37.3% malignant) was maintained in both training and test sets. This prevented biased evaluation and ensured metrics reflect real-world performance.

**No Missing Values/Duplicates:**
The clean dataset (no missing values, no duplicates) simplified preprocessing and allowed focus on model selection rather than data cleaning.

### Medical Implications

**False Positives (Benign predicted as Malignant):**
- **Consequence**: Unnecessary biopsies, patient anxiety, increased healthcare costs
- **Logistic Regression**: 2 false positives (precision 0.98)
- **KNN**: 3 false positives (precision 0.97)
- **Impact**: Logistic Regression causes fewer unnecessary procedures

**False Negatives (Malignant predicted as Benign):**
- **Consequence**: Missed cancer diagnosis, delayed treatment, worse patient outcomes, potential liability
- **Logistic Regression**: 5 false negatives (recall 0.95)
- **KNN**: 2 false negatives (recall 0.98)
- **Impact**: KNN misses fewer cancer cases

**Clinical Decision:**
The choice between models depends on the relative cost of false positives vs false negatives. In most clinical settings, false negatives are more serious (missed cancer can be fatal), so KNN's higher recall may be preferable. However, if false positives are causing significant harm (e.g., invasive biopsies with complications), Logistic Regression's higher precision may be preferable.

### When Logistic Regression Should Be Preferred

Logistic Regression should be preferred when:

1. **Interpretability is Critical**:
   - Doctors need to understand why a prediction was made
   - Regulatory requirements demand explainable models
   - Feature importance must be communicated to patients

2. **Prediction Speed is Important**:
   - Real-time decision support in clinical settings
   - High-volume screening programs
   - Mobile or edge computing applications

3. **False Positives are Costly**:
   - When follow-up procedures are invasive or expensive
   - When patient anxiety from false alarms is a major concern
   - When healthcare resources are limited

4. **High-Dimensional Data**:
   - When number of features is large
   - When feature selection is needed
   - When computational efficiency is important

5. **Linear Relationship is Reasonable**:
   - When domain knowledge suggests linear relationships
   - When simpler models are preferred for robustness

### When KNN Should Be Preferred

KNN should be preferred when:

1. **Maximizing Recall is Critical**:
   - When missing a positive case is unacceptable
   - When early detection is paramount
   - When false negatives are more serious than false positives

2. **Non-Linear Patterns are Expected**:
   - When domain knowledge suggests complex relationships
   - When linear models have failed previously
   - When decision boundaries are expected to be irregular

3. **Dataset is Small to Medium**:
   - When number of samples is < 10,000
   - When prediction speed is not critical
   - When memory is not a constraint

4. **Feature Dimensions are Low**:
   - When number of features is < 20
   - When curse of dimensionality is not a concern
   - When feature selection has already been performed

5. **Interpretability is Less Critical**:
   - When prediction accuracy is the primary goal
   - When model explanation is not required
   - When the model will be used as a black-box decision aid

## Question 9: Observations

**Observation:**
- Both models achieved excellent performance (>95% on all metrics)
- KNN achieved slightly better recall (0.98 vs 0.95), meaning fewer missed cancer diagnoses
- Logistic Regression achieved slightly better precision (0.98 vs 0.97), meaning fewer false alarms
- Logistic Regression is 20x faster at prediction time
- Logistic Regression provides interpretable feature importance
- The choice depends on clinical priorities: recall vs precision, speed vs accuracy, interpretability vs performance

**Interpretation:**
In breast cancer diagnosis, the slight performance advantage of KNN in recall may be clinically significant, as missing a cancer diagnosis can have serious consequences. However, Logistic Regression's interpretability and speed make it attractive for clinical deployment. The optimal choice depends on the specific clinical context and priorities.

**Practical Insight:**
In practice, a hybrid approach might be best: use Logistic Regression for routine screening (fast, interpretable), and use KNN as a second opinion for borderline cases (higher recall). Alternatively, ensemble methods combining both models could achieve the benefits of both. The key is aligning model selection with clinical priorities and patient outcomes.

## Self Learning

### Topic 1: Regularization in Logistic Regression

**Why Overfitting Occurs:**

Overfitting occurs when a model learns the training data too well, including noise and random fluctuations. In Logistic Regression, overfitting manifests as:
- Very large coefficient values
- High sensitivity to small changes in features
- Poor generalization to new data
- High variance in predictions

Overfitting is more likely when:
- Number of features is large relative to samples
- Features are highly correlated
- Training data is noisy

**L1 Regularization (Lasso):**

- **Mechanism**: Adds penalty equal to absolute value of coefficients
- **Formula**: Loss + lambda * sum(|coefficients|)
- **Effect**: Can drive some coefficients to exactly zero
- **Advantage**: Performs feature selection automatically
- **Disadvantage**: May be unstable with correlated features
- **When to use**: When feature selection is desired, when many features are irrelevant

**L2 Regularization (Ridge):**

- **Mechanism**: Adds penalty equal to square of coefficients
- **Formula**: Loss + lambda * sum(coefficients^2)
- **Effect**: Shrinks coefficients toward zero but not exactly zero
- **Advantage**: Stable with correlated features, unique solution
- **Disadvantage**: Does not perform feature selection
- **When to use**: When all features may be relevant, when features are correlated

**Difference Between L1 and L2:**

- **L1**: Creates sparse solutions (some coefficients = 0), good for feature selection
- **L2**: Creates dense solutions (all coefficients small but non-zero), good for handling correlated features
- **L1**: Geometric interpretation: diamond-shaped constraint region
- **L2**: Geometric interpretation: circular constraint region

**Penalty Parameter C:**

In sklearn, C is the inverse of regularization strength:
- **Large C** (e.g., C=10): Weak regularization, model can fit training data closely (risk of overfitting)
- **Small C** (e.g., C=0.01): Strong regularization, model coefficients are constrained (risk of underfitting)
- **Default**: C=1.0 (moderate regularization)

**Effect of Changing C:**

- **Increasing C**: Decreases regularization, allows larger coefficients, model becomes more complex (higher variance, lower bias)
- **Decreasing C**: Increases regularization, constrains coefficients, model becomes simpler (lower variance, higher bias)

**Relation with Bias and Variance:**

- **High C (low regularization)**: Low bias, high variance (complex model, risk of overfitting)
- **Low C (high regularization)**: High bias, low variance (simple model, risk of underfitting)
- **Optimal C**: Balances bias and variance for best generalization

**When to Use Regularization:**

- When number of features is large
- When features are correlated
- When training data is limited
- When model shows signs of overfitting (high training accuracy, low test accuracy)

### Topic 2: Weighted KNN

**Uniform Voting (Standard KNN):**

- **Mechanism**: All K nearest neighbors have equal vote in classification
- **Formula**: Class = majority vote among K neighbors
- **Assumption**: All neighbors are equally informative
- **Limitation**: Does not account for distance—far neighbors have same influence as near neighbors

**Distance Weighted Voting:**

- **Mechanism**: Closer neighbors have more influence than farther neighbors
- **Formula**: Weight = 1 / distance (or 1 / distance^2)
- **Implementation**: In sklearn, use `weights='distance'`
- **Advantage**: More intuitive—closer examples should be more similar

**Why Closer Neighbors Should Influence Prediction More:**

The fundamental assumption of KNN is that similar instances have similar labels. If a neighbor is closer in feature space, it is more similar to the query point and therefore more likely to have the same label. Giving closer neighbors more weight aligns with this intuition and typically improves classification accuracy.

**Example:**
- Query point: radius=15, texture=20
- Neighbor 1 (distance=0.5): Malignant
- Neighbor 2 (distance=5.0): Benign
- Neighbor 3 (distance=5.2): Benign

With uniform voting (K=3): 2 benign vs 1 malignant -> Predict Benign
With distance weighting: Malignant (very close) outweighs 2 distant benign -> Predict Malignant

**Using weights='distance':**

```python
knn_weighted = KNeighborsClassifier(n_neighbors=9, weights='distance')
knn_weighted.fit(X_train, y_train)
```

**Advantages of Weighted KNN:**

- **Better accuracy**: Typically improves classification by focusing on most similar cases
- **More intuitive**: Aligns with the principle that closer = more similar
- **Robust to K choice**: Less sensitive to the exact value of K
- **Handles noise**: Distant noisy points have less influence

**Limitations of Weighted KNN:**

- **Computational cost**: Still requires computing distances to all training points
- **Sensitive to outliers**: A single very close outlier can dominate the prediction
- **No free lunch**: May not always outperform uniform voting
- **Parameter tuning**: Still need to tune K and distance metric

**Real-World Medical Example:**

Consider diagnosing a new patient's breast mass based on historical cases:

- **Case A**: Very similar to the new case (distance=0.1 in feature space), was malignant
- **Case B**: Moderately similar (distance=2.0), was benign
- **Case C**: Moderately similar (distance=2.1), was benign

With uniform voting, the new case would be predicted benign (2 vs 1). However, Case A is extremely similar to the new case—much more similar than Cases B and C. With distance weighting, Case A's vote would carry much more weight, potentially leading to a malignant prediction.

In medical diagnosis, this is intuitive: a patient whose tumor characteristics are nearly identical to a known malignant case should be treated with high suspicion, even if there are several moderately similar benign cases. Distance weighting captures this intuition.

## Conclusion

This lab compared Logistic Regression and K-Nearest Neighbors classifiers on the Breast Cancer Wisconsin Diagnostic dataset. Key findings include:

**Dataset:**
The WDBC dataset contains 569 samples with 30 nuclear features computed from digitized images of breast masses. The dataset is clean (no missing values, no duplicates) with moderate class imbalance (62.7% benign, 37.3% malignant). Features vary widely in scale, making scaling essential.

**Preprocessing:**
Feature scaling was critical for both models. For Logistic Regression, scaling ensured optimization convergence and numerical stability. For KNN, scaling ensured all features contributed equally to distance calculations. Stratified train-test split maintained class distribution, enabling fair evaluation.

**Logistic Regression:**
Logistic Regression achieved excellent performance (accuracy 0.97, precision 0.98, recall 0.95, F1 0.96, ROC-AUC 0.99). It trained in approx 0.05 seconds and predicted in approx 0.001 seconds. The model provided interpretable coefficients showing that worst radius, worst perimeter, and mean concave points are the most important features for predicting malignancy.

**KNN:**
KNN achieved slightly better overall performance (accuracy 0.98, precision 0.97, recall 0.98, F1 0.97, ROC-AUC 0.99) with optimal K=9. It trained in approx 0.01 seconds (lazy learning) but predicted in approx 0.02 seconds (20x slower than Logistic Regression). The model achieved higher recall, meaning it caught more malignant cases.

**Comparison:**
Both models achieved excellent performance (>95% on all metrics). KNN had slightly better accuracy and recall, while Logistic Regression had slightly better precision and dramatically faster prediction time. Both had identical ROC-AUC scores (0.99), indicating excellent discriminative ability. The choice between models depends on application priorities: KNN for maximizing recall, Logistic Regression for speed and interpretability.

**Best Classifier:**
There is no clear "best" classifier—both models performed exceptionally well. Logistic Regression is preferable when prediction speed and interpretability are critical. KNN is preferable when maximizing recall (catching all cancer cases) is the priority. In clinical practice, the choice should be guided by the relative cost of false positives vs false negatives and the specific requirements of the clinical setting.

**Medical Relevance:**
In breast cancer diagnosis, both models are suitable for clinical use. Logistic Regression's interpretability allows doctors to understand and trust predictions, while KNN's higher recall may help catch more cancer cases. The slight performance differences are clinically meaningful: KNN's higher recall (0.98 vs 0.95) means 3 fewer missed cancer diagnoses per 100 patients, while Logistic Regression's higher precision (0.98 vs 0.97) means 1 fewer false alarm per 100 patients.

**Performance Metrics:**
Both models achieved ROC-AUC of 0.99, indicating excellent ability to distinguish between benign and malignant cases across all decision thresholds. The high precision (>0.97) means both models minimize false alarms, and the high recall (>0.95) means both models catch most cancer cases. These metrics suggest both models could be deployed in clinical settings with confidence.